In [1]:
from keras.optimizers import Adam
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, TerminateOnNaN, CSVLogger
from keras import backend as K
from keras.models import load_model
from keras.wrappers.scikit_learn import KerasClassifier
from math import ceil
import numpy as np
from matplotlib import pyplot as plt, patches
from sklearn.model_selection import train_test_split, cross_val_score

from models.ssd7_custom import build_model
from models.ssd7_resnet_backbone import resnet_build_model
from models.ssd300_custom import ssd300_build_model
from loss_function.custom_loss import AOILoss
from loss_function.custom_metric import class_mAP, offset_MAE
from custom_layers.GridCenters import GridCenters

from input_encoder_decoder.input_encoder import SSDInputEncoder
from input_encoder_decoder.output_decoder import decode_detections
from input_encoder_decoder.data_generator import DataGenerator

%matplotlib inline

2023-01-19 16:38:17.120140: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-01-19 16:38:17.693981: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: :/home/token/miniconda3/envs/tf/lib/
2023-01-19 16:38:17.694023: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer_plugin.so.7'; dlerror: libnvinfer_plugin.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: :/home/token/miniconda3/envs/tf/lib/
2023-01-19 16:38:17.694027: W tensorflow/compiler/tf2ten

In [2]:
img_height = 300 # Height of the input images
img_width = 300 # Width of the input images
img_channels = 3 # Number of color channels of the input images
intensity_mean = 127.5 # Set this to your preference (maybe `None`). The current settings transform the input pixel values to the interval `[-1,1]`.
intensity_range = 127.5 # Set this to your preference (maybe `None`). The current settings transform the input pixel values to the interval `[-1,1]`.
n_classes = 1 # Number of positive classes
normalize_coords = True # Whether or not the model is supposed to use coordinates relative to the image size
model_type = "resnet"

In [3]:
K.clear_session()

if model_type == "ssd7":

    model = build_model(image_size=(img_height, img_width, img_channels),
                        n_classes=n_classes,
                        l2_regularization=0.005,
                        normalize_coords=normalize_coords,
                        subtract_mean=intensity_mean,
                        divide_by_stddev=intensity_range)
    
elif model_type == "resnet":
    
    model = resnet_build_model(image_size=(img_height, img_width, img_channels),
                                n_classes=n_classes,
                                l2_regularization=0.5,
                                normalize_coords=normalize_coords)
    
elif model_type == "ssd300":
    
    model = ssd300_build_model(image_size=(img_height, img_width, img_channels),
                        n_classes=n_classes,
                        l2_regularization=0.005,
                        normalize_coords=normalize_coords,
                        subtract_mean=intensity_mean,
                        divide_by_stddev=intensity_range)
    

adam = Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.999, epsilon=1e-08)

aoi_loss = AOILoss(neg_pos_ratio=3, alpha=1.0)

model.compile(optimizer=adam, loss=aoi_loss.compute_loss, metrics=[class_mAP, offset_MAE])

2023-01-19 16:38:18.732408: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-01-19 16:38:18.735889: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-01-19 16:38:18.736023: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-01-19 16:38:18.736304: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorF

In [4]:
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_3 (InputLayer)           [(None, 300, 300, 3  0           []                               
                                )]                                                                
                                                                                                  
 vgg16 (Functional)             (None, 9, 9, 512)    14714688    ['input_3[0][0]']                
                                                                                                  
 bn0 (BatchNormalization)       (None, 9, 9, 512)    2048        ['vgg16[1][0]']                  
                                                                                                  
 elu0 (ELU)                     (None, 9, 9, 512)    0           ['bn0[1][0]']                

In [5]:
predictor_size = [model.get_layer('conv1').output_shape[1:3]]
print('Predictor Layer Dimensions: ', predictor_size)

encoder = SSDInputEncoder(img_height,
                          img_width,
                          n_classes,
                          predictor_sizes=predictor_size,
                          normalize_coords=True,
                          background_id=0)

generator = DataGenerator(parent_dir='/home/token/AOI_Project/Datasets/real_pcb_crops', encoder=encoder, augmentation=False, probability=0.1)

X, y = generator.get_data()

Predictor Layer Dimensions:  [(9, 9)]
Generating image arrays and encoding labels...
Converting images to arrays...


100%|█████████████████████████████████████| 8000/8000 [00:04<00:00, 1957.36it/s]


Images as numpy:
(8000, 300, 300, 3)
Parsing ground truth labels from .csv


100%|██████████████████████████████████████| 8000/8000 [00:08<00:00, 947.58it/s]


Unencoded labels:
8000
Encoded labels:
(8000, 81, 12)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

print('Train dataset: ', X_train.shape)
print('Test dataset: ', X_test.shape)
print('Train labels: ', y_train.shape)
print('Test labels: ', y_test.shape)

Train dataset:  (6400, 300, 300, 3)
Test dataset:  (1600, 300, 300, 3)
Train labels:  (6400, 81, 12)
Test labels:  (1600, 81, 12)


In [7]:
#test = offset_MAE(y_train[:5], y_train[:5])
#print(test)

In [8]:
model_checkpoint = ModelCheckpoint(filepath='checkpoints/ssd7_epoch-{epoch:02d}_loss-{loss:.4f}_val_loss-{val_loss:.4f}.h5',
                                   monitor='val_loss',
                                   verbose=1,
                                   save_best_only=True,
                                   save_weights_only=False,
                                   mode='auto',
                                   save_freq="epoch")

csv_logger = CSVLogger(filename='ssd7_training_log.csv',
                       separator=',',
                       append=True)

early_stopping = EarlyStopping(monitor='val_loss',
                               min_delta=0.001,
                               patience=8,
                               verbose=1,
                               restore_best_weights=True)

reduce_learning_rate = ReduceLROnPlateau(monitor='val_loss',
                                         factor=0.1,
                                         patience=3,
                                         verbose=1,
                                         min_delta=0.001,
                                         cooldown=0,
                                         min_lr=0.000001)

callbacks = [#model_checkpoint,
             #csv_logger,
             early_stopping,
             reduce_learning_rate]

In [9]:
batch_size = 16
initial_epoch   = 0
final_epoch     = 500
steps_per_epoch = 1000

history = model.fit(X_train,
                    y_train,
                    batch_size=batch_size,
                    #steps_per_epoch=steps_per_epoch,
                    epochs=final_epoch,
                    callbacks=callbacks,
                    validation_data=(X_test, y_test),
                    #validation_steps=ceil(X_test.shape[0]/batch_size),
                    initial_epoch=initial_epoch)

2023-01-19 16:38:33.271003: W tensorflow/tsl/framework/cpu_allocator_impl.cc:82] Allocation of 1728000000 exceeds 10% of free system memory.
2023-01-19 16:38:34.398763: W tensorflow/tsl/framework/cpu_allocator_impl.cc:82] Allocation of 1728000000 exceeds 10% of free system memory.


Epoch 1/500
Instructions for updating:
Lambda fuctions will be no more assumed to be used in the statement where they are used, or at least in the same block. https://github.com/tensorflow/tensorflow/issues/56089
Instructions for updating:
Use fn_output_signature instead


2023-01-19 16:38:36.550052: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:428] Loaded cuDNN version 8100
2023-01-19 16:38:36.988921: I tensorflow/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2023-01-19 16:38:36.989327: I tensorflow/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2023-01-19 16:38:36.989355: W tensorflow/compiler/xla/stream_executor/gpu/asm_compiler.cc:85] Couldn't get ptxas version string: INTERNAL: Couldn't invoke ptxas --version
2023-01-19 16:38:36.989816: I tensorflow/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2023-01-19 16:38:36.989859: W tensorflow/compiler/xla/stream_executor/gpu/redzone_allocator.cc:318] INTERNAL: Failed to launch ptxas
Relying on driver to perform ptx compilation. 
Modify $PATH to customize ptxas location.
This message will be only logged once.


400/400 [==============================] - 45s 97ms/step - loss: 132.2760 - class_mAP: 0.0679 - offset_MAE: 0.1678 - val_loss: 1.6069 - val_class_mAP: 0.0000e+00 - val_offset_MAE: 0.0557 - lr: 0.0010
Epoch 2/500
400/400 [==============================] - 38s 94ms/step - loss: 1.4478 - class_mAP: 0.1641 - offset_MAE: 0.0645 - val_loss: 1.3517 - val_class_mAP: 0.0019 - val_offset_MAE: 0.0449 - lr: 0.0010
Epoch 3/500
400/400 [==============================] - 38s 94ms/step - loss: 2.9127 - class_mAP: 0.1326 - offset_MAE: 0.1157 - val_loss: 11.6069 - val_class_mAP: 0.0000e+00 - val_offset_MAE: 0.4643 - lr: 0.0010
Epoch 4/500
400/400 [==============================] - 38s 95ms/step - loss: 3.2935 - class_mAP: 0.0544 - offset_MAE: 0.1014 - val_loss: 1.4633 - val_class_mAP: 0.0000e+00 - val_offset_MAE: 0.0441 - lr: 0.0010
Epoch 5/500
400/400 [==============================] - ETA: 0s - loss: 1.6839 - class_mAP: 0.1102 - offset_MAE: 0.0727
Epoch 5: ReduceLROnPlateau reducing learning rate to 0

KeyboardInterrupt: 

In [ ]:
plt.figure(figsize=(20,12))
plt.plot(history.history['loss'], label='loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.legend(loc='upper right', prop={'size': 24});

In [ ]:
plt.figure(figsize=(20,12))
plt.plot(history.history['class_mAP'], label='class_mAP')
plt.plot(history.history['val_class_mAP'], label='val_class_mAP')
plt.legend(loc='upper right', prop={'size': 24});

In [ ]:
plt.figure(figsize=(20,12))
plt.plot(history.history['offset_MAE'], label='offset_MAE')
plt.plot(history.history['val_offset_MAE'], label='val_offset_MAE')
plt.legend(loc='upper right', prop={'size': 24});

In [ ]:
predictions = model.predict(X_test[:50])
print(predictions.shape)

In [ ]:
decoded_pred = decode_detections(predictions, img_height=img_height, img_width=img_width)
print(decoded_pred[0])

In [ ]:
decoded_labels = decode_detections(y_test[:50], img_height=img_height, img_width=img_width)
print(decoded_labels[0])

In [ ]:
for i in range(len(decoded_pred[:20])):
    plt.figure(figsize=(10,6))
    plt.imshow(X_test[i])
    current_axis = plt.gca()

    colors = plt.cm.hsv(np.linspace(0, 1, n_classes+1)).tolist() # Set the colors for the bounding boxes
    classes = ['background', 'ic'] # Just so we can print class names onto the image instead of IDs
    
    for label in decoded_labels[i]:
        pred = np.array(label)
        corners = np.reshape(pred[2:], (-1, 2)).astype(int)
        color = colors[int(pred[0])]
        label = '{}'.format(classes[int(pred[0])])
        for points in corners:
            current_axis.add_patch(plt.Circle(tuple(points), 2, fill=False, color='blue'))
    
    for label in decoded_pred[i]:
        pred = np.array(label)
        corners = np.reshape(pred[2:], (-1, 2)).astype(int)
        color = colors[int(pred[0])]
        label = '{}: {:.2f}'.format(classes[int(pred[0])], pred[1])
        for points in corners:
            current_axis.add_patch(plt.Circle(tuple(points), 2, fill=False, color='red'))
        center_x = (corners[0, 0] + corners[3, 0]) / 2
        center_y = (corners[0, 1] + corners[3, 1]) / 2
        current_axis.text(center_x, center_y, label, size='x-small', color='white', bbox={'facecolor':color, 'alpha':1.0})
        

In [ ]:
#model.save("model_realistic_pcb_resnetv2_conv1")